# Zalo AI Challenge 2026 — LifeMate Offline Inference Pipeline (Colab T4 GPU)

Pipeline chạy **100% Offline** trên GPU T4 của Google Colab (Tuân thủ luật thi không dùng API bên ngoài):
- **ASR:** `vinai/PhoWhisper-large` (nhận dạng tiếng Việt chất lượng cao, xóa bỏ hiện tượng nghe nhầm từ)
- **SLM:** `Qwen/Qwen2.5-3B-Instruct` (mô hình <5B tối ưu cho Function Calling & JSON Schema)
- **Replay:** `zaic_tools.ToolEnv` xác thực schema trước khi gọi và tự động sinh `state_hash` chính xác

In [ ]:
# 1. Mount Google Drive và kiểm tra GPU T4
from google.colab import drive
drive.mount('/content/drive')

import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 * 1024 * 1024)
    print(f'VRAM: {vram_gb:.2f} GB')


In [ ]:
# 2. Cài đặt các thư viện cần thiết
!pip install -q transformers accelerate bitsandbytes scipy rapidfuzz


In [ ]:
# 3. Cấu hình đường dẫn dự án trên Google Drive
import sys, os, json, zipfile, io
from pathlib import Path

DRIVE_PROJECT = Path('/content/drive/MyDrive/lifemate')
DATA_DIR = DRIVE_PROJECT / 'data'
PUBLIC_TEST_DIR = DATA_DIR / 'public_test'
ZIP_PATH = DRIVE_PROJECT / 'train+public_test.zip'
CATALOG_PATH = PUBLIC_TEST_DIR / 'public_test_tools_catalog.json'
TEST_JSONL = PUBLIC_TEST_DIR / 'public_test.jsonl'
ASR_CACHE_FILE = DRIVE_PROJECT / 'asr_phowhisper_large.json'
SUBMISSION_FILE = DRIVE_PROJECT / 'submission.jsonl'

# Thêm zaic_tools vào sys.path
sys.path.insert(0, str(PUBLIC_TEST_DIR))
from zaic_tools import ToolEnv, schema_validate
print('ToolEnv and schema_validate loaded successfully!')


In [ ]:
# 4. Bước 1: Chạy ASR PhoWhisper-large trên GPU T4 (~2 phút cho 634 file)
import numpy as np
from scipy.io import wavfile
from transformers import pipeline

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Loading PhoWhisper-large on {device}...')
asr_pipe = pipeline(
    'automatic-speech-recognition',
    model='vinai/PhoWhisper-large',
    device=device,
    torch_dtype=torch.float16 if 'cuda' in device else torch.float32
)

with open(TEST_JSONL, 'r', encoding='utf-8') as f:
    samples = [json.loads(line) for line in f if line.strip()]

asr_cache = {}
if os.path.exists(ASR_CACHE_FILE):
    with open(ASR_CACHE_FILE, 'r', encoding='utf-8') as f:
        asr_cache = json.load(f)
    print(f'Loaded existing ASR cache: {len(asr_cache)} entries.')

to_transcribe = []
for s in samples:
    rel = s.get('audio', '')
    k = f'public_test/{rel}'
    if k not in asr_cache:
        to_transcribe.append(k)

print(f'Total samples: {len(samples)}, Need transcribe: {len(to_transcribe)}')

if to_transcribe:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        for idx, key in enumerate(to_transcribe, 1):
            candidates = [f'train+public_test/{key}', key]
            raw_bytes = None
            for c in candidates:
                try:
                    raw_bytes = zf.read(c)
                    break
                except KeyError:
                    pass
            if raw_bytes is None:
                asr_cache[key] = ''
                continue
            
            sr, a = wavfile.read(io.BytesIO(raw_bytes))
            if a.dtype == np.int16: a = a.astype(np.float32) / 32768.0
            elif a.dtype != np.float32: a = a.astype(np.float32)
            
            res = asr_pipe({'raw': a, 'sampling_rate': sr})
            asr_cache[key] = res.get('text', '').strip()
            
            if idx % 50 == 0 or idx == len(to_transcribe):
                print(f'Transcribed {idx}/{len(to_transcribe)}: {asr_cache[key][:50]}...')
                with open(ASR_CACHE_FILE, 'w', encoding='utf-8') as f:
                    json.dump(asr_cache, f, ensure_ascii=False, indent=2)

    with open(ASR_CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump(asr_cache, f, ensure_ascii=False, indent=2)
    print('ASR completed successfully!')


In [ ]:
# 5. Bước 2: Tải mô hình Qwen/Qwen2.5-3B-Instruct chạy Offline trên GPU T4
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = 'Qwen/Qwen2.5-3B-Instruct'
print(f'Loading {model_id} on GPU...')

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map='auto'
)
model.eval()
vram_used = torch.cuda.memory_allocated() / (1024 * 1024 * 1024)
print(f'Model loaded! VRAM allocated: {vram_used:.2f} GB')


In [ ]:
# 6. Bước 3: Định nghĩa Prompt Builder & Safe JSON Parsing
import re
from datetime import datetime, timedelta

with open(CATALOG_PATH, 'r', encoding='utf-8') as f:
    catalog_dict = json.load(f)

def filter_catalog(catalog_dict, installed_apps):
    filtered_tools = []
    for app in catalog_dict.get('apps', []):
        app_name = app.get('name', '')
        if app_name in installed_apps or app_name == 'sys':
            for tool in app.get('tools', []):
                filtered_tools.append({
                    'name': tool['name'],
                    'description': tool.get('description', ''),
                    'parameters': tool.get('parameters', {})
                })
    return filtered_tools

def build_system_prompt(filtered_tools, current_state):
    tools_str = json.dumps(filtered_tools, ensure_ascii=False, indent=1)
    compact_state = {k: v for k, v in current_state.items() if v}
    state_str = json.dumps(compact_state, ensure_ascii=False)
    
    return f'''Bạn là trợ lý AI LifeMate trong Zalo AI Challenge 2026.
Nhiệm vụ: Phân tích câu nói của người dùng và gọi các công cụ (tool_calls) phù hợp nhất.

CÔNG CỤ ĐƯỢC PHÉP DÙNG:
{tools_str}

DATABASE STATE HIỆN TẠI CỦA THIẾT BỊ:
{state_str}

QUY TẮC CỐT LÕI:
1. OUT-OF-SCOPE hoặc HỦY LỆNH ("thôi khỏi", "đừng làm gì"): trả về duy nhất mảng rỗng []
2. CÂU HỎI ĐIỀU KIỆN THỜI TIẾT ("Nếu mai mưa thì... không mưa thì thôi"): CHỈ gọi duy nhất `weather.get_forecast`.
3. KHÔNG ĐƯỢC TỰ BỊA ID: Lấy chính xác ID sự kiện/hóa đơn từ DATABASE STATE. Tham số hóa đơn là `source` (ví dụ "banking" hoặc "ewallet"). Tham số nhắc nhở là `content` và `time`.
4. ĐỊNH DẠNG ĐẦU RA: CHỈ trả về duy nhất một khối JSON array hợp lệ:
[
  {{"name": "tool.name", "arguments": {{"param": "val"}}}}
]'''

def build_user_prompt(sample, audio_text):
    ctx = sample.get('context', {})
    curr_time = ctx.get('current_time', '')
    user_prof = ctx.get('user_profile') or {}
    history = sample.get('history', [])
    
    history_str = ''
    if history:
        turns = [f"- {h.get('role')}: {h.get('audio_text') or h.get('text') or ''}" for h in history]
        history_str = '\nLịch sử hội thoại:\n' + '\n'.join(turns)
        
    return f'''Thời gian hiện tại: {curr_time}
User: {json.dumps(user_prof, ensure_ascii=False)}{history_str}
Người dùng vừa nói: "{audio_text}"
Hãy sinh JSON array các tool calls phù hợp:'''

def parse_json_calls(raw_output):
    text = raw_output.strip()
    m_code = re.search(r'```(?:json)?\s*(\[[\s\S]*?\])\s*```', text)
    if m_code: text = m_code.group(1).strip()
    else:
        m_arr = re.search(r'(\[[\s\S]*\])', text)
        if m_arr: text = m_arr.group(1).strip()
    try:
        data = json.loads(text)
        if isinstance(data, list):
            return [{'name': str(c['name']).strip(), 'arguments': c.get('arguments') or {}} for c in data if isinstance(c, dict) and 'name' in c]
    except Exception:
        pass
    return []


In [ ]:
# 7. Bước 4: Chạy suy luận Offline & Replay ToolEnv để sinh submission.jsonl (~10 phút trên T4)
import time
start_time = time.time()
results = []

print(f'Bắt đầu suy luận cho {len(samples)} mẫu...')
with open(SUBMISSION_FILE, 'w', encoding='utf-8') as f_out:
    for idx, s in enumerate(samples, 1):
        rel = s.get('audio', '')
        text = asr_cache.get(f'public_test/{rel}', '')
        
        env = ToolEnv('public_test', s['id'], root=DATA_DIR)
        
        t_low = text.lower()
        oos = any(w in t_low for w in ['khách sạn', 'hotel', 'homestay', 'resort', 'hộ chiếu', 'visa', 'bảo hiểm', 'karaoke', 'spa'])
        cancel = any(w in t_low for w in ['thôi khỏi', 'đừng làm gì', 'không cần nữa']) and not any(w in t_low for w in ['chỉ cần xem', 'xem giúp'])
        
        if not oos and not cancel:
            installed = set(s.get('context', {}).get('installed_apps', {}).keys())
            filtered = filter_catalog(catalog_dict, installed)
            sys_p = build_system_prompt(filtered, env.get_state())
            usr_p = build_user_prompt(s, text)
            
            messages = [
                {'role': 'system', 'content': sys_p},
                {'role': 'user', 'content': usr_p}
            ]
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer([prompt], return_tensors='pt').to(model.device)
            
            with torch.inference_mode():
                out_ids = model.generate(
                    **inputs,
                    max_new_tokens=384,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )
                gen_ids = out_ids[0][inputs.input_ids.shape[1]:]
                resp_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
            
            pred_calls = parse_json_calls(resp_text)
            for c in pred_calls:
                val = env.validate(c['name'], c.get('arguments') or {})
                if val.get('ok'):
                    env.call(c['name'], c.get('arguments') or {})
        
        row = {
            'id': s['id'],
            'tool_calls': env.get_tool_calls(),
            'state_hash': env.get_state_hash()
        }
        results.append(row)
        f_out.write(json.dumps(row, ensure_ascii=False) + '\n')
        f_out.flush()
        
        if idx % 20 == 0 or idx == len(samples):
            elapsed = time.time() - start_time
            print(f'[{idx}/{len(samples)}] - {elapsed/60:.1f} mins elapsed...')

print(f'XONG! File kết quả đã ghi tại: {SUBMISSION_FILE}')


In [ ]:
# 8. Bước 5: Validate file submission.jsonl trước khi nộp
with open(SUBMISSION_FILE, 'r', encoding='utf-8') as f:
    sub_lines = [json.loads(line) for line in f if line.strip()]

assert len(sub_lines) == len(samples), f'Số dòng không khớp: {len(sub_lines)}/{len(samples)}'

empty_cnt = sum(1 for r in sub_lines if not r['tool_calls'])
non_empty_cnt = len(sub_lines) - empty_cnt

print(f'Tổng số mẫu nộp: {len(sub_lines)}')
print(f'Số mẫu có gọi tool: {non_empty_cnt} ({non_empty_cnt/len(sub_lines)*100:.1f}%)')
print(f'Số mẫu rỗng: {empty_cnt} ({empty_cnt/len(sub_lines)*100:.1f}%)')

all_valid = True
for r in sub_lines:
    if not schema_validate('public_test', r['id'], r['tool_calls'], root=DATA_DIR):
        print(f'LỖI SCHEMA: Mẫu {r["id"]} không hợp lệ!')
        all_valid = False
        break

if all_valid:
    print('CHÚC MỪNG: File submission.jsonl đã vượt qua 100% kiểm tra hợp lệ của BTC!')
